十、学习练习
实现一个异步文本流生成器。
将流式片段拼接为完整答案。
为订单意图定义 Pydantic 模型。
模拟缺字段、错误枚举和非法 JSON。
为 FastAPI 添加 SSE 接口，并处理客户端取消。

In [ ]:
pip install fastapi uvicorn

In [12]:
from collections.abc import AsyncIterator
from pydantic import BaseModel
from enum import Enum
import json
import asyncio
from fastapi.responses import StreamingResponse
from fastapi import FastAPI
import uvicorn

class IntentType(str, Enum):
    CREATE_ORDER = "create_order"
    UPDATE_ORDER = "update_order"
    CANCEL_ORDER = "cancel_order"


class OrderIntent(BaseModel):
    intent: IntentType
    order_id: str

async def text_gen() -> AsyncIterator[OrderIntent]:
    buffer = ""
    async for text in text_stream():
        buffer += text
        print(f"Received text: {text}")
    print(f"Final buffer: {buffer}")
    data = json.loads(buffer)
    intent = OrderIntent.model_validate(data)
    yield intent.model_dump_json()



async def text_stream() -> AsyncIterator[str]:
    try:
        yield "{\"intent\": \"create_order\", "
        yield " \"order_id\": \"12345\"}"
    except asyncio.CancelledError as e:
        print(f"Text generation was cancelled: {e}")
        raise
    finally:
        print("Text generation completed.todo close stream")

app = FastAPI(title="FastAPI Task API")

@app.get("/chat")
async def startup_event() -> StreamingResponse:
    return StreamingResponse(text_gen(), media_type="application/json")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()

# json = await text_gen()
# print("Text generation completed.")

INFO:     Started server process [11160]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:62222 - "GET /chat HTTP/1.1" 200 OK
Received text: {"intent": "create_order", 
Received text:  "order_id": "12345"}
Text generation completed.todo close stream
Final buffer: {"intent": "create_order",  "order_id": "12345"}
INFO:     127.0.0.1:62222 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [11160]
